# Analisis salarial

Objetivo: describir la estructura de compensacion de `data/salarios.csv` (1000 empleados), medir brechas por departamento y proponer ajustes.

**Supuestos**
- Salario = `salario_mensual_cop` (COP mensual).
- **Salario alto**: salario >= percentil 90 de toda la nomina.
- **Referencia**: mediana salarial de la misma `categoria` (controla el nivel jerarquico).
- **Brecha de departamento** = mediana del salario del departamento / mediana de la referencia de sus empleados - 1. Negativa = paga por debajo de lo esperado para su mezcla de categorias.
- **Umbral de accion**: brecha menor a -5 % se recomienda ajuste; mayor a +5 % se monitorea.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
OUT = ROOT / "submission"
OUT.mkdir(exist_ok=True)
df = pd.read_csv(ROOT / "data" / "salarios.csv")
print(df.shape, "nulos:", int(df.isna().sum().sum()), "ids duplicados:", int(df.employee_id.duplicated().sum()))
df.describe().round(0)

(1000, 7) nulos: 0 ids duplicados: 0


,experiencia_tecnica_anios,salario_mensual_cop
count,1000.0,1000.0
mean,9.0,10073236.0
std,5.0,4585479.0
min,0.0,2101179.0
25%,6.0,6229443.0
50%,9.0,9459118.0
75%,12.0,13280637.0
max,25.0,21465350.0


## Estructura salarial por categoria

In [2]:
por_cat = df.groupby("categoria").salario_mensual_cop.agg(["count", "median", "mean", "min", "max"]).sort_values("median")
print(por_cat.round(0).to_string())
print("Correlacion experiencia-salario:", round(df.experiencia_tecnica_anios.corr(df.salario_mensual_cop), 3))

              count      median        mean       min       max
categoria                                                      
Analista        140   4182974.0   4269913.0   2101179   7031261
Profesional     254   6343129.0   6432287.0   3962617  10606182
Especialista    254   9703956.0   9657929.0   7176087  12520551
Senior          232  13521626.0  13641340.0  10825527  17353049
Principal       120  18437684.0  18531183.0  15690191  21465350
Correlacion experiencia-salario: 0.857


## Salario alto por trayectoria y categoria

In [3]:
umbral = df.salario_mensual_cop.quantile(0.90)
df["salario_alto"] = df.salario_mensual_cop >= umbral
high = (df.groupby(["trayectoria", "categoria"])
          .agg(empleados=("employee_id", "count"), salario_alto=("salario_alto", "sum"))
          .reset_index())
high["pct_salario_alto"] = (high.salario_alto / high.empleados * 100).round(1)
high["umbral_cop"] = round(umbral)
high = high.sort_values("pct_salario_alto", ascending=False)
high.to_csv(OUT / "high_salary_by_career.csv", index=False)
print(f"Umbral P90: {umbral:,.0f} COP")
high

Umbral P90: 17,340,830 COP


,trayectoria,categoria,empleados,salario_alto,pct_salario_alto,umbral_cop
1,Directiva,Principal,53,53,100.0,17340830
6,Técnica,Principal,67,45,67.2,17340830
3,Directiva,Senior,88,2,2.3,17340830
0,Directiva,Especialista,46,0,0.0,17340830
2,Directiva,Profesional,18,0,0.0,17340830
4,Técnica,Analista,140,0,0.0,17340830
5,Técnica,Especialista,208,0,0.0,17340830
7,Técnica,Profesional,236,0,0.0,17340830
8,Técnica,Senior,144,0,0.0,17340830


## Brecha por departamento (mediana vs referencia de categoria)

In [4]:
df["referencia"] = df.groupby("categoria").salario_mensual_cop.transform("median")
gap = (df.groupby(["gerencia", "departamento"])
         .agg(empleados=("employee_id", "count"), mediana_salario=("salario_mensual_cop", "median"),
              mediana_referencia=("referencia", "median"))
         .reset_index())
gap["brecha_pct"] = ((gap.mediana_salario / gap.mediana_referencia - 1) * 100).round(2)
gap = gap.sort_values("brecha_pct")
gap.to_csv(OUT / "department_pay_gap.csv", index=False)
gap

,gerencia,departamento,empleados,mediana_salario,mediana_referencia,brecha_pct
1,Comercial,Clientes menores,124,8107724.0,9703956.0,-16.45
4,Operaciones,Operaciones,185,9481048.0,9703956.0,-2.30
2,Finanzas,Planeación financiera,112,9753459.5,9703956.0,0.51
6,Talento,Recursos humanos,120,9780085.0,9703956.0,0.78
5,Operaciones,Tecnología,156,10012159.0,9703956.0,3.18
0,Comercial,Clientes mayoristas,173,10342126.0,9703956.0,6.58
3,Finanzas,Tesorería,130,7128684.0,6343129.0,12.38


## Recomendaciones de compensacion

In [5]:
bajo = df[df.salario_mensual_cop < 0.8 * df.referencia].groupby("departamento").size()
rec = gap[["gerencia", "departamento", "empleados", "brecha_pct"]].copy()
rec["empleados_bajo_80pct_ref"] = rec.departamento.map(bajo).fillna(0).astype(int)
def accion(b):
    if b < -5: return "Ajuste salarial prioritario"
    if b > 5: return "Monitorear (por encima de la referencia)"
    return "Mantener"
rec["recomendacion"] = rec.brecha_pct.map(accion)
rec["ajuste_mediano_sugerido_cop"] = ((-rec.brecha_pct.clip(upper=0) / 100) * gap.mediana_referencia.values).round(0)
rec = rec.sort_values("brecha_pct")
rec.to_csv(OUT / "compensation_recommendations.csv", index=False)
rec

,gerencia,departamento,empleados,brecha_pct,empleados_bajo_80pct_ref,recomendacion,ajuste_mediano_sugerido_cop
1,Comercial,Clientes menores,124,-16.45,20,Ajuste salarial prioritario,1596301.0
4,Operaciones,Operaciones,185,-2.30,6,Mantener,223191.0
2,Finanzas,Planeación financiera,112,0.51,1,Mantener,-0.0
6,Talento,Recursos humanos,120,0.78,5,Mantener,-0.0
5,Operaciones,Tecnología,156,3.18,0,Mantener,-0.0
0,Comercial,Clientes mayoristas,173,6.58,1,Monitorear (por encima de la referencia),-0.0
3,Finanzas,Tesorería,130,12.38,19,Monitorear (por encima de la referencia),-0.0


## Conclusiones

In [6]:
peor, mejor = gap.iloc[0], gap.iloc[-1]
cat_alta = high.groupby("categoria").salario_alto.sum().sort_values(ascending=False)
dir_ = df.groupby("trayectoria").salario_mensual_cop.median()
conc = pd.DataFrame([
    ("Empleados analizados", len(df), "Nomina completa sin nulos ni duplicados"),
    ("Mediana salarial (COP)", round(df.salario_mensual_cop.median()), "Mediana de toda la nomina"),
    ("Umbral salario alto P90 (COP)", round(umbral), "Salario alto = >= P90"),
    ("Empleados con salario alto", int(df.salario_alto.sum()), f"Categoria con mas casos: {cat_alta.index[0]} ({cat_alta.iloc[0]})"),
    ("Mediana Directiva vs Tecnica", round(dir_["Directiva"] / dir_["Técnica"], 2), "Razon de medianas por trayectoria"),
    ("Correlacion experiencia-salario", round(df.experiencia_tecnica_anios.corr(df.salario_mensual_cop), 3), "Pearson, experiencia tecnica en anios"),
    ("Departamento mas rezagado", peor.departamento, f"Brecha {peor.brecha_pct:.2f} % ({peor.gerencia})"),
    ("Departamento mas favorecido", mejor.departamento, f"Brecha {mejor.brecha_pct:.2f} % ({mejor.gerencia})"),
    ("Departamentos con ajuste prioritario", int((rec.recomendacion == "Ajuste salarial prioritario").sum()), "Brecha menor a -5 %"),
    ("Empleados bajo 80 % de su referencia", int(rec.empleados_bajo_80pct_ref.sum()), "Riesgo de retencion"),
], columns=["metrica", "valor", "detalle"])
conc.to_csv(OUT / "analysis_conclusions.csv", index=False)
conc

,metrica,valor,detalle
0,Empleados analizados,1000,Nomina completa sin nulos ni duplicados
1,Mediana salarial (COP),9459118,Mediana de toda la nomina
2,Umbral salario alto P90 (COP),17340830,Salario alto = >= P90
3,Empleados con salario alto,100,Categoria con mas casos: Principal (98)
4,Mediana Directiva vs Tecnica,1.8,Razon de medianas por trayectoria
5,Correlacion experiencia-salario,0.857,"Pearson, experiencia tecnica en anios"
6,Departamento mas rezagado,Clientes menores,Brecha -16.45 % (Comercial)
7,Departamento mas favorecido,Tesorería,Brecha 12.38 % (Finanzas)
8,Departamentos con ajuste prioritario,1,Brecha menor a -5 %
9,Empleados bajo 80 % de su referencia,52,Riesgo de retencion


Lectura: la categoria explica la mayor parte de la dispersion salarial; las brechas por departamento se miden contra la mediana de su categoria, asi que reflejan desvio de politica y no mezcla de cargos. Los departamentos con brecha negativa mayor al umbral son candidatos a revision, priorizando los de mayor numero de empleados bajo 80 % de la referencia. Las cifras exactas estan en la tabla de arriba y en `submission/analysis_conclusions.csv`.